In [1]:
from pynq import Overlay

pedal_overlay = Overlay('/home/xilinx/jupyter_notebooks/Pedal/dma_axis_ip_example.bit')
audio_ip = pedal_overlay.pedal_top_0          

dma = pedal_overlay.axi_dma
dma_send = pedal_overlay.axi_dma.sendchannel
dma_recv = pedal_overlay.axi_dma.recvchannel
switch_ip = pedal_overlay.axi_gpio_0

switches_offset = 0x10

CONTROL_REGISTER = 0x0
audio_ip.write(CONTROL_REGISTER, 0x81)


In [ ]:
from pynq import Overlay

pedal_overlay = Overlay('/home/xilinx/jupyter_notebooks/Pedal/dma_axis_ip_example.bit')

In [2]:
pedal_overlay?


In [ ]:
from pynq import PL
PL.reset() # This clears the current state of the Programmable Logic server


In [ ]:
# def update_switches():
sw_val = switch_ip.read()
print(sw_val)

In [22]:
import numpy as np
from pynq import allocate
from scipy.io import wavfile
import time

sample_rate, data = wavfile.read('/home/xilinx/jupyter_notebooks/Pedal/freesound_community-simple-guitar-melody-102329.wav')
# print(sample_rate)
data_size = data.shape[0]
input_buffer = allocate(shape=(data_size,), dtype=np.int16)
samples_for_200ms = int(sample_rate * 0.2)
# print(samples_for_200ms)
# input_buffer.fill(1)

np.copyto(input_buffer, data)

chunk_size_bytes = 16000
bytes_sent = 0
total_bytes = input_buffer.nbytes
# print(total_bytes)

def update_switches():
    sw_val = switch_ip.read()
    audio_ip.write(switches_offset, sw_val)
    return sw_val
    
    
output_buffer = allocate(shape=(data_size,), dtype=np.int16) 
start_time = time.time() 
print(update_switches())
while bytes_sent < total_bytes:
    end = min(bytes_sent + chunk_size_bytes, total_bytes)
#     if(bytes_sent % 100 == 0) : 
#         print(bytes_sent)
    # Send segment
    dma_send.transfer(input_buffer[bytes_sent:end])
    
    # Receive corresponding segment
    dma_recv.transfer(output_buffer[bytes_sent:end])
    
    # Wait for completion
    dma_send.wait()
    dma_recv.wait()

    bytes_sent = end


final_output = output_buffer.astype(np.int16)
# print(final_output)
# print(output_buffer[0:34])
end_time = time.time() # Capture end time

wavfile.write('processed_output.wav', sample_rate, final_output)
execution_time = end_time - start_time
print(f"Total processing time: {execution_time:.6f} seconds")


15
Total processing time: 0.038304 seconds
